# Skin Lesion Bias Reduction — Colab training

Runs the EfficientNetV2-B0 baseline classifier on a Colab GPU using the project code in `src/` and a Fitzpatrick17k dataset you've already uploaded.

**Order of operations**
1. Confirm GPU + mount Drive (if used)
2. Point the notebook at your code + data
3. Install dependencies
4. Train the baseline (with class weights, unfreeze schedule, save-best-by-val-loss)
5. Evaluate the best checkpoint and render a markdown bias report
6. *Optional* — kick off the cGAN training (placeholder, run later)

## 1. Verify GPU and Colab environment

In [1]:
import sys

IN_COLAB = "google.colab" in sys.modules
print("Colab:", IN_COLAB)

import torch
print("torch:", torch.__version__, "cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

Colab: False
torch: 2.10.0 cuda available: False


In [17]:
!nvidia-smi

Mon Apr 27 06:16:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   36C    P0             55W /  400W |       6MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Connect your code and data

Two common layouts work:

- **Drive layout** — repo and dataset both sit under `MyDrive`. Mount Drive and point `PROJECT_ROOT` at the repo there. Outputs persist between sessions.
- **Local Colab layout** — clone the repo into `/content/` and put the dataset under `/content/dataset/`. Faster I/O, but everything is wiped when the runtime ends.

Edit `PROJECT_ROOT`, `DATASET_CSV`, and `IMAGE_DIR` in the cell below to match where you uploaded things.

In [18]:
# Mount Drive only if you're using the Drive layout. Skip this cell otherwise.
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from pathlib import Path

# === EDIT THESE ===
PROJECT_ROOT = Path("/content/drive/MyDrive/SkinLesionBiasReduction")  # repo root containing src/, dataset/, run_*.sh
# IMAGE_DIR    = PROJECT_ROOT / "dataset/images"
IMAGE_DIR    = PROJECT_ROOT / "dataset/images"

DATASET_CSV = PROJECT_ROOT / "dataset/fitzpatrick17k_cleaned.csv"
# ==================

print("PROJECT_ROOT:", PROJECT_ROOT)
print("IMAGE_DIR:   ", IMAGE_DIR,   "exists:", IMAGE_DIR.exists())
assert PROJECT_ROOT.exists(), f"PROJECT_ROOT does not exist: {PROJECT_ROOT}"
assert IMAGE_DIR.exists(),    f"IMAGE_DIR does not exist:    {IMAGE_DIR}"

%cd $PROJECT_ROOT

PROJECT_ROOT: /content/drive/MyDrive/SkinLesionBiasReduction
IMAGE_DIR:    /content/drive/MyDrive/SkinLesionBiasReduction/dataset/images exists: True
/content/drive/MyDrive/SkinLesionBiasReduction


In [20]:
import subprocess

DRIVE_IMAGE_DIR = IMAGE_DIR
LOCAL_IMAGE_DIR = Path("/content/local_images")

def _count_files(d):
    r = subprocess.run(f"ls -1 '{d}' 2>/dev/null | wc -l", shell=True, capture_output=True, text=True)
    return int(r.stdout.strip())

n_source = _count_files(DRIVE_IMAGE_DIR)
n_local  = _count_files(LOCAL_IMAGE_DIR) if LOCAL_IMAGE_DIR.exists() else 0
print(f"Drive: {n_source} files | Local cache: {n_local} files")

if n_local >= n_source > 0:
    print("Local cache is complete — skipping copy.")
else:
    print("Copying dataset to local SSD via tar (streaming, a few minutes)...")
    subprocess.run(f"rm -rf '{LOCAL_IMAGE_DIR}'", shell=True)
    subprocess.run(
        f"cd '{DRIVE_IMAGE_DIR}' && tar cf - . | "
        f"(mkdir -p '{LOCAL_IMAGE_DIR}' && cd '{LOCAL_IMAGE_DIR}' && tar xf -)",
        shell=True, check=True,
    )
    n_local = _count_files(LOCAL_IMAGE_DIR)
    assert n_local >= n_source, f"Copy incomplete: expected {n_source}, got {n_local}"
    print(f"Done — {n_local} files copied to {LOCAL_IMAGE_DIR}")

IMAGE_DIR = LOCAL_IMAGE_DIR
print(f"IMAGE_DIR → {IMAGE_DIR}")

Drive: 16518 files | Local cache: 0 files
Copying dataset to local SSD via tar (streaming, a few minutes)...


KeyboardInterrupt: 

In [ ]:
# Alternative: if you don't have the repo on Drive, clone it into /content/ and copy your dataset in.
# Uncomment, replace the URL with your fork, and re-run cell `configure-paths` with PROJECT_ROOT=/content/SkinLesionBiasReduction.
# !git clone git@github.com:hoangnam310/SkinLesionBiasReduction.git

In [ ]:
#!git fetch origin main && git pull origin main

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 3.43 KiB | 1024 bytes/s, done.
From https://github.com/hoangnam310/SkinLesionBiasReduction
 * branch            main       -> FETCH_HEAD
   64d51ef..f18ed1a  main       -> origin/main
From https://github.com/hoangnam310/SkinLesionBiasReduction
 * branch            main       -> FETCH_HEAD
Updating 64d51ef..f18ed1a
^C


## 3. Install dependencies

Colab images already include `torch`, `torchvision`, `numpy`, `pandas`, `Pillow`, `tqdm`, `matplotlib`, and `scipy`. The trainer additionally needs **timm** and **scikit-learn** (sklearn is usually preinstalled, timm usually is not). Tensorboard is optional and is also usually preinstalled.

In [ ]:
!pip install --quiet timm 'scikit-learn>=1.3'

## 4. Train the baseline classifier

Full training at **224×224** for 40 epochs. The backbone is frozen for the first 3 epochs (head-only warmup), then unfrozen for fine-tuning at a lower LR.

Key flags being used:
- `--class_weights` — inverse-frequency weighted CrossEntropyLoss; the single biggest fix from the prior run's bias analysis.
- `--freeze_backbone --unfreeze_epoch 3 --fine_tune_lr 1e-5` — three warmup epochs on the head, then full fine-tune.
- The trainer also auto-saves the best checkpoint by val_loss across all epochs (no extra flag needed).

> **Tip:** For a quick smoke test, set `EPOCHS = 1` and `IMAGE_SIZE = 64` to verify the pipeline works before committing to the full run.

In [ ]:
IMAGE_SIZE   = 224
EPOCHS       = 40
BATCH_SIZE   = 32        # 32 fits comfortably on A100 at 224x224; use 16 on a T4.
LR           = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS  = 4

OUTPUT_DIR = PROJECT_ROOT / "outputs/baseline_efficientnet"
print("Outputs will land under:", OUTPUT_DIR)

Outputs will land under: /content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet


In [ ]:
!python src/train_baseline_efficientnet.py \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{IMAGE_DIR}" \
    --image_size {IMAGE_SIZE} \
    --epochs {EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --lr {LR} \
    --weight_decay {WEIGHT_DECAY} \
    --num_workers {NUM_WORKERS} \
    --class_weights \
    --freeze_backbone --unfreeze_epoch 3 --fine_tune_lr 1e-5 \
    --output_dir "{OUTPUT_DIR}" \
    --device cuda

Using device: cuda
Image size: 224x224
Train batches: 399, Val batches: 100
Pretrained: True
Freeze backbone: True
Training classifier head only (backbone frozen).
Class weights (benign, malignant, non-neoplastic): [2.4567, 2.4956, 0.4562]
Epoch [1/40] train_loss=1.0420, train_acc=0.4895, val_loss=1.0077, val_acc=0.5265  ← best
Epoch [2/40] train_loss=0.9842, train_acc=0.5554, val_loss=0.9713, val_acc=0.5591  ← best
Epoch [3/40] train_loss=0.9558, train_acc=0.5705, val_loss=0.9504, val_acc=0.5713  ← best
Backbone unfrozen at epoch 3; continuing fine-tuning with lr=1e-05
Epoch [4/40] train_loss=0.9226, train_acc=0.5866, val_loss=0.9100, val_acc=0.5954  ← best
Epoch [5/40] train_loss=0.8769, train_acc=0.6135, val_loss=0.8795, val_acc=0.5929  ← best
Epoch [6/40] train_loss=0.8397, train_acc=0.6386, val_loss=0.8449, val_acc=0.6224  ← best
Epoch [7/40] train_loss=0.7976, train_acc=0.6635, val_loss=0.8191, val_acc=0.6509  ← best
Epoch [8/40] train_loss=0.7606, train_acc=0.6806, val_loss=0.79

## 5. Evaluate the best checkpoint

`train_baseline_efficientnet.py` restores best-by-val-loss weights before the final eval, so the auto-generated `metrics.json` already uses that snapshot. We additionally re-run `evaluate.py` to write `logs/evaluation_metrics.json` (with bias breakdowns) and then render a markdown report.

In [ ]:
runs = sorted(OUTPUT_DIR.glob("*/checkpoint.pt"), key=lambda p: p.stat().st_mtime, reverse=True)
assert runs, f"No checkpoint.pt under {OUTPUT_DIR}"
LATEST_CKPT = runs[0]
print("Latest checkpoint:", LATEST_CKPT)

Latest checkpoint: /content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet/20260427_055032/checkpoint.pt


In [ ]:
!python src/evaluate.py \
    --checkpoint "{LATEST_CKPT}" \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{IMAGE_DIR}" \
    --batch_size {BATCH_SIZE} \
    --num_workers {NUM_WORKERS} \
    --device cuda \
    --logs_dir "{PROJECT_ROOT / 'logs'}"

Using device: cuda
Loading checkpoint from /content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet/20260427_055032/checkpoint.pt
Evaluating on 3191 samples (100 batches)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 741, in __next__
    data = self._next_data()
           ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1524, in _next_data
    idx, data = self._get_data()
                ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1473, in _get_data
    success, data = self._try_get_data()
                    ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1310, in _try_get_data
    data = self._data_queue.get(timeout=timeout)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/queue.py", line 180

In [ ]:
!python src/metrics_report.py \
    --json "{PROJECT_ROOT / 'logs' / 'evaluation_metrics.json'}"

Wrote /content/drive/MyDrive/SkinLesionBiasReduction/logs/20260427_043404_report.md


In [ ]:
from IPython.display import Markdown, display

run_name = LATEST_CKPT.parent.name
report_path = PROJECT_ROOT / "logs" / f"{run_name}_report.md"
print("Report:", report_path)
display(Markdown(report_path.read_text()))

Report: /content/drive/MyDrive/SkinLesionBiasReduction/logs/20260427_043404_report.md


# Evaluation Report

| Field | Value |
| --- | --- |
| Checkpoint | `/content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet/20260427_043404/checkpoint.pt` |
| Split | val |
| Image size | 64 |
| CSV path | `/content/drive/MyDrive/SkinLesionBiasReduction/dataset/fitzpatrick17k_cleaned.csv` |
| Image dir | `/content/drive/MyDrive/SkinLesionBiasReduction/dataset/images` |
| Generated at | 2026-04-27T04:39:51.277323 |

## Global metrics

| Metric | Value |
| --- | --- |
| Samples | 3191 |
| Top-1 Accuracy | 0.3811 |
| Macro AUROC | 0.5553 |
| Macro AUPRC | 0.3711 |

## Per-Fitzpatrick subgroup

| Fitzpatrick | n | Accuracy | Macro AUROC | Macro AUPRC |
| --- | --- | --- | --- | --- |
| 1 | 602 | 0.3571 | 0.5571 | 0.3750 |
| 2 | 920 | 0.3587 | 0.5642 | 0.3864 |
| 3 | 665 | 0.3805 | 0.5441 | 0.3694 |
| 4 | 554 | 0.4134 | 0.5340 | 0.3647 |
| 5 | 318 | 0.4119 | 0.5011 | 0.3424 |
| 6 | 132 | 0.4394 | 0.6384 | 0.4349 |

### Per-class AUROC by Fitzpatrick (one-vs-rest)

| Fitzpatrick | benign (n / AUROC) | malignant (n / AUROC) | non-neoplastic (n / AUROC) |
| --- | --- | --- | --- |
| 1 | 101 / 0.5256 | 85 / 0.6047 | 416 / 0.5409 |
| 2 | 112 / 0.5192 | 164 / 0.6190 | 644 / 0.5543 |
| 3 | 91 / 0.5011 | 100 / 0.6068 | 474 / 0.5245 |
| 4 | 75 / 0.5309 | 59 / 0.5498 | 420 / 0.5212 |
| 5 | 39 / 0.4383 | 29 / 0.5496 | 250 / 0.5155 |
| 6 | 8 / 0.6956 | 16 / 0.6185 | 108 / 0.6011 |

### Per-class AUPRC by Fitzpatrick (one-vs-rest)

| Fitzpatrick | benign (n / AUPRC) | malignant (n / AUPRC) | non-neoplastic (n / AUPRC) |
| --- | --- | --- | --- |
| 1 | 101 / 0.1971 | 85 / 0.2089 | 416 / 0.7189 |
| 2 | 112 / 0.1370 | 164 / 0.2792 | 644 / 0.7429 |
| 3 | 91 / 0.1595 | 100 / 0.2182 | 474 / 0.7305 |
| 4 | 75 / 0.1400 | 59 / 0.1779 | 420 / 0.7761 |
| 5 | 39 / 0.1051 | 29 / 0.1100 | 250 / 0.8120 |
| 6 | 8 / 0.2461 | 16 / 0.1751 | 108 / 0.8836 |

## Classification metrics

| Metric | Value |
| --- | --- |
| Accuracy | 0.3811 |
| Balanced accuracy | 0.3895 |
| Macro F1 | 0.3257 |
| Weighted F1 | 0.4297 |

### Per-class

| Class | Precision | Recall | F1 | Support |
| --- | --- | --- | --- | --- |
| benign | 0.1424 | 0.3286 | 0.1987 | 426 |
| malignant | 0.1959 | 0.4658 | 0.2758 | 453 |
| non-neoplastic | 0.7648 | 0.3741 | 0.5025 | 2312 |

### Confusion matrix

| True \ Pred | benign | malignant | non-neoplastic |
| --- | --- | --- | --- |
| benign | 140 | 156 | 130 |
| malignant | 106 | 211 | 136 |
| non-neoplastic | 737 | 710 | 865 |


## 6. Segmentation experiments — train classifiers on cv2- and SAM2-segmented images

To isolate the effect of segmentation-aware preprocessing, run two more classifiers with the **same hyperparameters** as section 4 — only `--image_dir` changes:

- **cv2 variant** — ROI selection driven by LAB color variation, no SAM model. Fast.
- **SAM2 variant** — ROI selection constrained to SAM2's foreground mask (segmentation-aware crop).

Both are produced at **224×224** so the trainer's transform doesn't have to upscale at load time (which throws away mid-frequency texture detail).

Workflow:
1. Configure paths and segmentation knobs
2. Build the cv2-segmented dir
3. Install SAM2 + download a checkpoint
4. Build the SAM2-segmented dir
5. (Optional) Backfill SAM2 failures with cv2 so all three runs see the same md5s
6. Common training args
7. Train cv2 → train SAM2
8. Evaluate both

In [ ]:
# Local-disk dirs for fast IO during training; mirrored back to Drive at the end.
LOCAL_CV2_DIR  = Path("/content/local_images_cv2_224")
LOCAL_SAM2_DIR = Path("/content/local_images_sam2_224")
DRIVE_CV2_DIR  = PROJECT_ROOT / "dataset/images_cv2_224"
DRIVE_SAM2_DIR = PROJECT_ROOT / "dataset/images_sam2_224"

# Segmentation knobs (apply to both cv2 and SAM2)
SEG_SIZE  = 384      # SAM + ROI search resolution (downscaled to OUT_SIZE after)
OUT_SIZE  = 224      # final image side; matches the trainer's --image_size
CROP_FRAC = 0.6
MIN_SKIN  = 0.85

(PROJECT_ROOT / "logs").mkdir(parents=True, exist_ok=True)
for d in (LOCAL_CV2_DIR, LOCAL_SAM2_DIR, DRIVE_CV2_DIR, DRIVE_SAM2_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("CV2  →", DRIVE_CV2_DIR)
print("SAM2 →", DRIVE_SAM2_DIR)
print("Source images at:", LOCAL_IMAGE_DIR, f"({_count_files(LOCAL_IMAGE_DIR)} files)")

### 6.1 cv2 segmentation (no SAM, fast)

Builds `images_cv2_224/` from the local cache. Runs in a few minutes on CPU. Resumable.

In [ ]:
!python src/preprocess_segmentation.py \
    --backend cv2 \
    --strategy color \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{LOCAL_IMAGE_DIR}" \
    --output_dir "{LOCAL_CV2_DIR}" \
    --seg_size {SEG_SIZE} \
    --out_size {OUT_SIZE} \
    --crop_frac {CROP_FRAC} \
    --min_skin {MIN_SKIN} 2>&1 | tee "{PROJECT_ROOT / 'logs' / 'preprocess_cv2_224.log'}"

# Mirror to Drive so it survives a Colab disconnect
!mkdir -p "{DRIVE_CV2_DIR}" && rsync -a "{LOCAL_CV2_DIR}/" "{DRIVE_CV2_DIR}/"
print("cv2 dir:",
      _count_files(LOCAL_CV2_DIR), "files local /",
      _count_files(DRIVE_CV2_DIR), "on Drive")

### 6.2 Install SAM2 and fetch the tiny checkpoint

`sam2` is not preinstalled on Colab. The checkpoint and config name must match — `sam2_hiera_tiny.pt` pairs with `sam2_hiera_t.yaml` (the config string is resolved by name inside the sam2 package).

In [ ]:
SAM2_CKPT_DIR  = PROJECT_ROOT / "checkpoints"
SAM2_CKPT_PATH = SAM2_CKPT_DIR / "sam2_hiera_tiny.pt"
SAM2_CKPT_DIR.mkdir(parents=True, exist_ok=True)

if not SAM2_CKPT_PATH.exists():
    !curl -L -o "{SAM2_CKPT_PATH}" \
        https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_tiny.pt
size_mb = SAM2_CKPT_PATH.stat().st_size / 1e6 if SAM2_CKPT_PATH.exists() else 0
print(f"SAM2 checkpoint: {SAM2_CKPT_PATH} (exists={SAM2_CKPT_PATH.exists()}, {size_mb:.1f} MB)")

# Install SAM2 itself (Meta's repo). Skip if already installed in this runtime.
try:
    import sam2  # noqa: F401
    print("sam2 already installed")
except ImportError:
    !pip install --quiet "git+https://github.com/facebookresearch/sam2.git"
    import sam2  # noqa: F401
    print("sam2 installed")

### 6.3 SAM2 segmentation (GPU)

Run the preprocessor with `--backend sam2`. ~30–45 min on a T4, ~10–15 min on an A100 for 16,519 images at `seg_size=384`. Resumable — files already in `--output_dir` are skipped.

The `tee` pipe captures the per-failure `[fail] <md5>: <exc>` lines, so you can audit which images SAM2 dropped this run.

In [ ]:
!python src/preprocess_segmentation.py \
    --backend sam2 \
    --strategy color \
    --sam2_checkpoint "{SAM2_CKPT_PATH}" \
    --sam2_config sam2_hiera_t.yaml \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{LOCAL_IMAGE_DIR}" \
    --output_dir "{LOCAL_SAM2_DIR}" \
    --seg_size {SEG_SIZE} \
    --out_size {OUT_SIZE} \
    --crop_frac {CROP_FRAC} \
    --min_skin {MIN_SKIN} \
    --device cuda 2>&1 | tee "{PROJECT_ROOT / 'logs' / 'preprocess_sam2_224.log'}"

!mkdir -p "{DRIVE_SAM2_DIR}" && rsync -a "{LOCAL_SAM2_DIR}/" "{DRIVE_SAM2_DIR}/"
print("SAM2 dir:",
      _count_files(LOCAL_SAM2_DIR), "files local /",
      _count_files(DRIVE_SAM2_DIR), "on Drive")

### 6.4 (Optional but recommended) Backfill SAM2 failures with cv2

`SkinLesionDataset` silently drops md5s whose `.jpg` is missing in `--image_dir`. If SAM2 fails on N images, the SAM2 trainer sees N fewer rows than the cv2 / raw trainers, and the seed-42 split produces a different cohort — breaking the comparison.

This cell re-runs the preprocessor with `--backend cv2` against the **same** `--output_dir`. Files already produced by SAM2 are skipped (resume-mode default), so this only fills in the gaps. After it finishes, `LOCAL_SAM2_DIR` should match `LOCAL_CV2_DIR`'s file count.

In [ ]:
!python src/preprocess_segmentation.py \
    --backend cv2 \
    --strategy color \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{LOCAL_IMAGE_DIR}" \
    --output_dir "{LOCAL_SAM2_DIR}" \
    --seg_size {SEG_SIZE} \
    --out_size {OUT_SIZE} \
    --crop_frac {CROP_FRAC} \
    --min_skin {MIN_SKIN} 2>&1 | tee -a "{PROJECT_ROOT / 'logs' / 'preprocess_sam2_224.log'}"

!rsync -a "{LOCAL_SAM2_DIR}/" "{DRIVE_SAM2_DIR}/"
print("SAM2 dir after backfill:",
      _count_files(LOCAL_SAM2_DIR), "files local /",
      _count_files(DRIVE_SAM2_DIR), "on Drive")

### 6.5 Common training args

These match the section 4 baseline run (`20260427_055032`) **exactly**. The only thing that changes between the three runs is `--image_dir`.

In [ ]:
# Mirror section 4 — keep these in sync if you change section 4.
SEG_IMAGE_SIZE   = 224
SEG_EPOCHS       = 40
SEG_BATCH_SIZE   = 32
SEG_LR           = 1e-4
SEG_WEIGHT_DECAY = 1e-4
SEG_NUM_WORKERS  = 4
SEG_OUTPUT_DIR   = OUTPUT_DIR     # all three runs share this parent; each gets its own timestamped subdir
print("Outputs land under:", SEG_OUTPUT_DIR)

### 6.6 Train classifier on cv2-segmented images

Identical hyperparameters to the section 4 baseline (`20260427_055032`); only `--image_dir` changes.

In [ ]:
!python src/train_baseline_efficientnet.py \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{LOCAL_CV2_DIR}" \
    --image_size {SEG_IMAGE_SIZE} \
    --epochs {SEG_EPOCHS} \
    --batch_size {SEG_BATCH_SIZE} \
    --lr {SEG_LR} \
    --weight_decay {SEG_WEIGHT_DECAY} \
    --num_workers {SEG_NUM_WORKERS} \
    --class_weights \
    --freeze_backbone --unfreeze_epoch 3 --fine_tune_lr 1e-5 \
    --output_dir "{SEG_OUTPUT_DIR}" \
    --device cuda

### 6.7 Train classifier on SAM2-segmented images

Same args as above, just `--image_dir` points at `LOCAL_SAM2_DIR`.

In [ ]:
!python src/train_baseline_efficientnet.py \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{LOCAL_SAM2_DIR}" \
    --image_size {SEG_IMAGE_SIZE} \
    --epochs {SEG_EPOCHS} \
    --batch_size {SEG_BATCH_SIZE} \
    --lr {SEG_LR} \
    --weight_decay {SEG_WEIGHT_DECAY} \
    --num_workers {SEG_NUM_WORKERS} \
    --class_weights \
    --freeze_backbone --unfreeze_epoch 3 --fine_tune_lr 1e-5 \
    --output_dir "{SEG_OUTPUT_DIR}" \
    --device cuda

### 6.8 Evaluate both segmented runs

Each cell below picks the most recent run whose `args.image_dir` matches the segmented dir, then writes `logs/<run_name>_report.md` and renders it inline. Run the cv2 cell first, then the SAM2 cell — `evaluate.py` overwrites `logs/evaluation_metrics.json` each call, but the per-run markdown report is keyed by run name so both are kept.

In [ ]:
import torch as _torch_for_seg

def _seg_latest_under(image_dir: Path, label: str) -> Path:
    """Pick the most recent run under SEG_OUTPUT_DIR whose args.image_dir == image_dir."""
    runs = sorted(SEG_OUTPUT_DIR.glob("*/checkpoint.pt"),
                  key=lambda p: p.stat().st_mtime, reverse=True)
    for ckpt in runs:
        try:
            args = _torch_for_seg.load(ckpt, map_location="cpu", weights_only=False).get("args", {})
        except Exception:
            continue
        if Path(args.get("image_dir", "")) == image_dir:
            print(f"[{label}] {ckpt}")
            return ckpt
    raise FileNotFoundError(f"No checkpoint found for image_dir={image_dir}")

CV2_CKPT = _seg_latest_under(LOCAL_CV2_DIR, "cv2")

!python src/evaluate.py \
    --checkpoint "{CV2_CKPT}" \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{LOCAL_CV2_DIR}" \
    --batch_size {SEG_BATCH_SIZE} \
    --num_workers {SEG_NUM_WORKERS} \
    --device cuda \
    --logs_dir "{PROJECT_ROOT / 'logs'}"

!python src/metrics_report.py \
    --json "{PROJECT_ROOT / 'logs' / 'evaluation_metrics.json'}"

cv2_report = PROJECT_ROOT / "logs" / f"{CV2_CKPT.parent.name}_report.md"
print("cv2 report:", cv2_report)
display(Markdown(cv2_report.read_text()))

In [ ]:
SAM2_CKPT = _seg_latest_under(LOCAL_SAM2_DIR, "sam2")

!python src/evaluate.py \
    --checkpoint "{SAM2_CKPT}" \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{LOCAL_SAM2_DIR}" \
    --batch_size {SEG_BATCH_SIZE} \
    --num_workers {SEG_NUM_WORKERS} \
    --device cuda \
    --logs_dir "{PROJECT_ROOT / 'logs'}"

!python src/metrics_report.py \
    --json "{PROJECT_ROOT / 'logs' / 'evaluation_metrics.json'}"

sam2_report = PROJECT_ROOT / "logs" / f"{SAM2_CKPT.parent.name}_report.md"
print("SAM2 report:", sam2_report)
display(Markdown(sam2_report.read_text()))

## 7. Quick smoke test (optional)

If you want to verify the pipeline before committing to a full 40-epoch run, temporarily override section **4** with:

```python
IMAGE_SIZE = 64
EPOCHS     = 1
BATCH_SIZE = 128
```

Then re-run sections **4** and **5**. Once it completes without errors, restore the defaults (224 / 40 / 32) and launch the real training.

## 8. Optional — train the cGAN later

The generative side lives in `src/train.py` (vanilla cGAN; the WGAN-GP critic exists in `src/cgan.py` but the WGAN trainer hasn't been committed yet). To run on Colab once you're ready:

In [ ]:
# Uncomment when you want to start cGAN training. Outputs land under outputs/<timestamp>/.
# !python src/train.py \
#     --csv_path "{DATASET_CSV}" \
#     --image_dir "{IMAGE_DIR}" \
#     --epochs 200 \
#     --batch_size 64 \
#     --lr 0.0002 \
#     --output_dir "{PROJECT_ROOT / 'outputs'}" \
#     --device cuda

In [ ]:
# After (or during) cGAN training, watch losses + sample grids in TensorBoard:
# %load_ext tensorboard
# %tensorboard --logdir $PROJECT_ROOT/outputs